# Premiers pas avec les *embeddings* de phrases

Un modèle d'*embeddings* transforme un texte en un vecteur, de sorte que deux textes de sens proche aient des vecteurs proches. Le modèle de ces travaux pratiques : [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), multilingue, 1024 dimensions, l'un des meilleurs modèles ouverts de sa taille.

1. Comparer une douzaine de phrases
2. Chercher dans trois semaines de presse, avec et sans instruction
3. Chercher dans une autre langue
4. Cosinus, produit scalaire, distance euclidienne
5. Tronquer les vecteurs (Matryoshka)

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même : le seul calcul trop long pour un processeur, les vecteurs des 3 950 passages de l'exercice 2, n'y est fait que sur 8 passages, puis remplacé par son résultat calculé sur GPU.

## Données et modèle

Les passages de la presse de 1914 (voir le TP précédent), leurs vecteurs calculés sur GPU (pour une exécution sans GPU), et le modèle, chargé en demi-précision sur GPU : deux fois moins de mémoire, plus rapide.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/lab-results/month-embeddings.npy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# Sur CPU, float32 : les demi-précisions (float16, et bfloat16, celle des poids publiés,
# qu'on aurait par défaut) y sont très lentes.
DTYPE = torch.float16 if GPU else torch.float32
print("GPU" if GPU else "Pas de GPU : les calculs longs chargeront leur résultat")

modele = SentenceTransformer(
  "Qwen/Qwen3-Embedding-0.6B", device=DEVICE, model_kwargs={"dtype": DTYPE}
)
# Le modèle accepte 32 000 tokens : on coupe à 512, assez pour nos passages, sinon
# les plus longs saturent la mémoire du GPU.
modele.max_seq_length = 512

## 1. Comparer une douzaine de phrases

Encodez les phrases de `PHRASES` avec `modele.encode_document`, calculez la matrice de leurs similarités avec `modele.similarity`, et affichez-la avec `plt.imshow` (les phrases en étiquettes des axes). Quelles paires obtiennent les meilleurs scores ? Trouvez deux surprises.

**Documentation utile**

- [`matplotlib.pyplot.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)
- [`Axes.set_xticks`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_xticks.html)

In [ ]:
PHRASES = [
  "Le VPN se déconnecte sans arrêt",
  "Impossible de me connecter au réseau depuis chez moi",
  "My VPN keeps disconnecting",
  "J'ai reçu un e-mail suspect",
  "Un message frauduleux est arrivé dans ma boîte",
  "L'imprimante n'imprime plus",
  "Mon écran reste noir",
  "Comment poser des congés ?",
  "Quand arrive la fiche de paie ?",
  "Le serveur fonctionne",
  "Le serveur ne fonctionne pas",
  "La cantine ferme à 14 h",
]

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele.encode_document(PHRASES)`, puis `modele.similarity(vecteurs, vecteurs).numpy()` : la matrice carrée des similarités
- `fig, ax = plt.subplots(figsize=(9, 8))`, `image = ax.imshow(similarites)`, puis `fig.colorbar(image)` pour la légende des couleurs
- `ax.set_xticks(range(n), PHRASES, rotation=90)` : les positions, puis les étiquettes (tournées à 90°) ; de même pour `ax.set_yticks`, sans rotation

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
vecteurs = ...  # les vecteurs des phrases
similarites = modele.similarity(..., ...).numpy()  # la matrice des similarités

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(...)
ax.set_xticks(range(len(PHRASES)), PHRASES, rotation=90)
ax.set_yticks(range(len(PHRASES)), ...)
fig.colorbar(image)
plt.show()
```

### Solution

In [ ]:
vecteurs = modele.encode_document(PHRASES)
similarites = modele.similarity(vecteurs, vecteurs).numpy()

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(similarites, cmap="viridis")
ax.set_xticks(range(len(PHRASES)), PHRASES, rotation=90)
ax.set_yticks(range(len(PHRASES)), PHRASES)
fig.colorbar(image)
plt.show()

# - Une phrase et sa traduction anglaise sont parmi les paires les plus proches (0,82) : le
#   modèle est multilingue.
# - Mais « Le serveur fonctionne » et « Le serveur ne fonctionne pas », de sens opposé, le
#   sont plus encore (0,89, la paire la plus proche) : les embeddings captent le sujet bien
#   mieux qu'une négation.
# - Deux phrases sans rapport ne tombent pas à 0 : seule la comparaison des scores a un sens.

## 2. Chercher dans trois semaines de presse

`mois` contient les passages publiés entre le 20 juillet et le 10 août 1914 (cellule suivante). Encodez leurs textes, `textes`, avec `modele.encode_document` (sans GPU, `textes` ne contient que les 8 premiers : l'exercice suivant charge les vecteurs de tous, calculés sur GPU).

Écrivez ensuite une fonction `cherche(requete, k=5, instruction=True)` qui affiche la date, le score et le début des `k` passages les plus proches de la requête. Avec `instruction=True`, encodez la requête avec `modele.encode_query`, qui la précède d'une instruction (« *Given a web search query, retrieve relevant passages that answer the query* ») ; sinon avec `modele.encode`, sans instruction.

**Documentation utile**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
passages = pd.read_parquet("passages.parquet")
# Les passages du 20 juillet au 10 août 1914, numérotés à partir de 0 : la ligne d'un passage
# dans mois est aussi la position de son vecteur.
mois = passages[passages.date.between("1914-07-20", "1914-08-10")].reset_index(
  drop=True
)
print(len(mois), "passages")
# Tous les textes sur GPU ; sans GPU, les 8 premiers (une minute environ).
textes = mois.text.tolist() if GPU else mois.text[:8].tolist()

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele.encode_document(textes, batch_size=32, show_progress_bar=True)` : les vecteurs des passages, par lots de 32, avec une barre de progression
- `modele.encode_query(requete)` avec l'instruction, `modele.encode(requete)` sans
- `modele.similarity(vecteur, vecteurs_mois)[0].numpy()` : un score par passage
- `np.argsort(-scores)[:k]` : les positions des `k` meilleurs (on trie les scores opposés pour avoir les meilleurs en tête)
- `f"{mois.date[i]:%d/%m} {scores[i]:.3f} {mois.text[i][:150]}"` : la date, le score et le début du passage `i`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
vecteurs_mois = modele.encode_document(
  ..., batch_size=32, show_progress_bar=True
)  # les textes : textes


def cherche(requete: str, k: int = 5, instruction: bool = True) -> None:
  vecteur = modele.encode_query(requete) if instruction else modele.encode(...)
  scores = modele.similarity(vecteur, vecteurs_mois)[0].numpy()  # un score par passage
  for i in ...:  # les positions des k meilleurs scores
    print(f"{mois.date[i]:%d/%m} {scores[i]:.3f} {mois.text[i][:150]}")
```

### Solution

In [ ]:
vecteurs_mois = modele.encode_document(textes, batch_size=32, show_progress_bar=True)


def cherche(requete: str, k: int = 5, instruction: bool = True) -> None:
  vecteur = modele.encode_query(requete) if instruction else modele.encode(requete)
  scores = modele.similarity(vecteur, vecteurs_mois)[0].numpy()
  for i in np.argsort(-scores)[:k]:
    print(f"{mois.date[i]:%d/%m} {scores[i]:.3f} {mois.text[i][:150]}")

### Trois recherches

Cherchez, avec et sans instruction :

- l'assassinat de Jean Jaurès
- la mobilisation générale
- le verdict du procès de Mme Caillaux

Les résultats sont-ils pertinents ? L'instruction change-t-elle quelque chose ?

In [ ]:
# Sans GPU, l'exercice précédent n'a encodé que 8 passages : on charge les vecteurs des
# 3 950, calculés sur GPU par le même code.
if not GPU:
  vecteurs_mois = np.load("month-embeddings.npy").astype(np.float32)
print(vecteurs_mois.shape)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `cherche(requete, instruction=True)`, puis `instruction=False`, pour chacune des trois requêtes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for requete in (
  "l'assassinat de Jean Jaurès",
  "la mobilisation générale",
  "le verdict du procès de Mme Caillaux",
):
  for instruction in (True, False):
    print(f"--- {requete} ({'avec' if instruction else 'sans'} instruction)")
    cherche(...)
```

#### Solution

In [ ]:
for requete in (
  "l'assassinat de Jean Jaurès",
  "la mobilisation générale",
  "le verdict du procès de Mme Caillaux",
):
  for instruction in (True, False):
    print(f"--- {requete} ({'avec' if instruction else 'sans'} instruction)")
    cherche(requete, instruction=instruction)

# Requête par requête :
# - Jaurès : avec et sans instruction, les cinq passages rapportent l'assassinat ou les
#   hommages qui suivent ; l'instruction ne change que leur ordre.
# - La mobilisation générale : avec l'instruction, les cinq passages parlent d'une
#   mobilisation (belge, autrichienne, française, russe) ; sans, quatre, et une convocation
#   socialiste hors sujet. Le début affiché ne montre pas toujours la partie pertinente :
#   la mobilisation belge du premier passage vient plus loin.
# - Le verdict (l'acquittement, le 28 juillet au soir) : sans instruction, le compte rendu
#   de la fin du procès, qui l'annonce (29/07), vient en tête ; avec, il recule au 4e rang.
# L'instruction aide sur une requête et nuit sur une autre : ne concluez pas sur trois
# exemples, un jeu de requêtes jugées tranche (TP suivant). Et ses scores ne se comparent
# pas à ceux d'un encodage sans instruction : on compare les scores d'une même requête.

## 3. Chercher dans une autre langue

Le corpus est en français, le modèle multilingue : une requête et sa traduction devraient trouver les mêmes passages. Écrivez une fonction `premiers(requete, k=10)` qui renvoie les positions des `k` passages les plus proches de la requête, encodée avec l'instruction. Pour chacune des trois requêtes de l'exercice précédent et sa traduction en anglais (« *the assassination of Jean Jaurès* », « *the general mobilization* », « *the verdict of Mme Caillaux's trial* »), mesurez la part de passages communs à leurs 10 premiers. Et pourquoi pas dans une autre langue que vous connaissez ?

Critère de réussite : la requête sur Jaurès et sa traduction ont les mêmes 10 premiers passages (100 %). Les autres paires en sont-elles aussi près ? Pour celle qui s'en écarte le plus, affichez ses 15 premiers passages dans les deux langues avec `cherche`, et comparez leurs scores.

**Documentation utile**

- [Ensembles (`set`)](https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `premiers` : comme dans `cherche`, `modele.encode_query(requete)` puis `modele.similarity(vecteur, vecteurs_mois)[0].numpy()`, et `np.argsort(-scores)[:k]` à renvoyer
- `set(premiers(requete)) & set(premiers(traduction))` : les passages communs aux deux classements ; leur nombre divisé par 10 donne la part
- `cherche(requete, k=15)` : les 15 premiers passages, avec leur score

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def premiers(requete: str, k: int = 10) -> np.ndarray:
  vecteur = modele.encode_query(requete)
  scores = modele.similarity(vecteur, vecteurs_mois)[0].numpy()
  return ...  # les positions des k meilleurs scores


for requete, traduction in (
  ("l'assassinat de Jean Jaurès", "the assassination of Jean Jaurès"),
  ("la mobilisation générale", "the general mobilization"),
  ("le verdict du procès de Mme Caillaux", "the verdict of Mme Caillaux's trial"),
):
  communs = ...  # les passages communs aux deux classements
  print(f"{len(communs) / 10:4.0%} de passages communs : {requete} / {traduction}")

cherche(..., k=15)  # la paire la plus loin de 100 %, en français
cherche(..., k=15)  # puis en anglais
```

### Solution

In [ ]:
def premiers(requete: str, k: int = 10) -> np.ndarray:
  vecteur = modele.encode_query(requete)
  scores = modele.similarity(vecteur, vecteurs_mois)[0].numpy()
  return np.argsort(-scores)[:k]


for requete, traduction in (
  ("l'assassinat de Jean Jaurès", "the assassination of Jean Jaurès"),
  ("la mobilisation générale", "the general mobilization"),
  ("le verdict du procès de Mme Caillaux", "the verdict of Mme Caillaux's trial"),
  ("l'assassinat de Jean Jaurès", "die Ermordung von Jean Jaurès"),
):
  communs = set(premiers(requete)) & set(premiers(traduction))
  print(f"{len(communs) / 10:4.0%} de passages communs : {requete} / {traduction}")

cherche("la mobilisation générale", k=15)
cherche("the general mobilization", k=15)

# Jaurès 100 %, Caillaux 90 %, la mobilisation 60 % (et Jaurès en allemand, 70 %) : une
# requête et sa traduction tombent presque au même endroit, le modèle est multilingue. Pour
# la mobilisation, les trois premiers passages sont les mêmes, dans le même ordre ; les
# écarts viennent plus bas, où les scores se suivent de près (moins de 0,01 du 8e au 14e
# passage en français) : un rien les réordonne. Un nom propre (Jaurès, Caillaux) ancre la
# requête dans les deux langues ; la mobilisation a beaucoup de passages presque aussi bons.

## 4. Cosinus, produit scalaire, distance euclidienne

Calculez la norme des vecteurs de `vecteurs_mois` (`np.linalg.norm`). Puis, pour la requête « la mobilisation générale », classez les passages selon le cosinus, le produit scalaire et la distance euclidienne (du plus proche au plus loin) : les trois classements ont-ils les mêmes 10 premiers ? Pourquoi ? Quand ne serait-ce plus le cas ?

**Documentation utile**

- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)
- [`numpy.ndarray.astype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `vecteurs_mois.astype(np.float32)` : sur GPU, le modèle est en demi-précision ; on calcule en `float32`
- `np.linalg.norm(vecteurs, axis=1)` : la norme de chaque ligne
- produit scalaire : `vecteurs @ vecteur_requete` ; cosinus : le produit scalaire divisé par les normes des deux vecteurs ; distance euclidienne : `np.linalg.norm(vecteurs - vecteur_requete, axis=1)`
- `np.argsort(-scores)[:10]` pour le produit scalaire et le cosinus (les plus grands d'abord), `np.argsort(distance)[:10]` pour la distance (la plus petite d'abord)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
vecteurs = vecteurs_mois.astype(np.float32)
print(np.linalg.norm(vecteurs, axis=1)[:5])

vecteur_requete = modele.encode_query("la mobilisation générale").astype(np.float32)
produit = vecteurs @ vecteur_requete
cosinus = produit / (np.linalg.norm(vecteurs, axis=1) * np.linalg.norm(...))
distance = np.linalg.norm(..., axis=1)
classements = {
  "cosinus": np.argsort(-cosinus)[:10],
  "produit scalaire": np.argsort(...)[:10],
  "euclidienne": np.argsort(...)[:10],
}
for nom, classement in classements.items():
  print(f"{nom:17}", classement)
```

### Solution

In [ ]:
vecteurs = vecteurs_mois.astype(np.float32)
print(np.linalg.norm(vecteurs, axis=1)[:5])

vecteur_requete = modele.encode_query("la mobilisation générale").astype(np.float32)
produit = vecteurs @ vecteur_requete
cosinus = produit / (np.linalg.norm(vecteurs, axis=1) * np.linalg.norm(vecteur_requete))
distance = np.linalg.norm(vecteurs - vecteur_requete, axis=1)
classements = {
  "cosinus": np.argsort(-cosinus)[:10],
  "produit scalaire": np.argsort(-produit)[:10],
  "euclidienne": np.argsort(distance)[:10],
}
for nom, classement in classements.items():
  print(f"{nom:17}", classement)

# Le modèle normalise ses vecteurs (norme 1) : le produit scalaire est alors le cosinus, et
# |u - v|² = 2 - 2 cos(u, v), donc la distance euclidienne classe dans le même ordre. Avec des
# vecteurs de normes différentes, le produit scalaire favoriserait les plus longs.

## 5. Tronquer les vecteurs (Matryoshka)

Le modèle est entraîné pour que les **premières** dimensions portent l'essentiel du sens. Tronquez les vecteurs des passages et des requêtes à 512, 256, 128 et 64 dimensions (puis renormalisez-les), et mesurez, pour 200 passages pris au hasard comme requêtes, la part de leurs 10 plus proches voisins (en 1024 dimensions) qu'on retrouve après troncature. Faites de même avec la quantification binaire, qui garde les 1024 dimensions mais seulement leur signe (un bit chacune). Combien d'octets prend un vecteur à chaque taille, et combien de mémoire faudrait-il pour 1 million de passages ?

**Documentation utile**

- [Matryoshka](https://sbert.net/examples/sentence_transformer/training/matryoshka/README.html)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)
- [`numpy.random.Generator.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html)
- [`numpy.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `x[:, :dims]` garde les `dims` premières colonnes ; on renormalise en divisant par `np.linalg.norm(x, axis=1, keepdims=True)` (`keepdims=True` : une norme par ligne, dans une forme qui permet de diviser la matrice)
- `np.random.default_rng(0).choice(len(vecteurs), 200, replace=False)` : 200 positions au hasard, sans remise
- les 10 plus proches voisins : `x[requetes] @ x.T`, avec `-np.inf` pour le passage lui-même (qui n'est pas son propre voisin), puis `np.argsort(-scores, axis=1)[:, :10]`
- la part retrouvée : l'intersection des voisins avant et après troncature, divisée par 10, en moyenne sur les requêtes
- `np.where(vecteurs > 0, 1.0, -1.0)` : le signe de chaque dimension ; ces vecteurs ont tous la même norme, le produit scalaire les classe donc comme le cosinus, sans renormaliser
- la mémoire : `dims * 4` octets par vecteur en `float32`, `1024 / 8` pour un bit par dimension (une fois les signes rangés en bits), fois 1 million
- le modèle sait aussi tronquer ses vecteurs lui-même, avec `truncate_dim` à son chargement, sans les renormaliser

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def tronque(x: np.ndarray, dims: int) -> np.ndarray:
  x = x[:, :dims]
  return x / np.linalg.norm(..., axis=1, keepdims=True)


rng = np.random.default_rng(0)
requetes = rng.choice(len(vecteurs), 200, replace=False)


def voisins(x: np.ndarray) -> np.ndarray:
  scores = x[requetes] @ x.T
  scores[
    np.arange(len(requetes)), requetes
  ] = ...  # un passage n'est pas son propre voisin
  return np.argsort(..., axis=1)[:, :10]


def compare(nom: str, x: np.ndarray, octets: float) -> None:
  trouves = voisins(x)
  part = np.mean(
    [... for a, b in zip(reference, trouves)]
  )  # la part de voisins retrouvés
  print(
    f"{nom:18} : {part:4.0%} des voisins retrouvés, {octets:4.0f} octets par vecteur,"
    f" {octets * 1e6 / 1e9:.1f} Go par million de passages"
  )


reference = voisins(vecteurs)
compare("1024 dimensions", vecteurs, 1024 * 4)
for dims in (512, 256, 128, 64):
  compare(f"{dims} dimensions", tronque(vecteurs, dims), ...)  # octets par vecteur
compare("1024 bits (signes)", ..., 1024 / 8)  # le signe de chaque dimension
```

### Solution

In [ ]:
def tronque(x: np.ndarray, dims: int) -> np.ndarray:
  x = x[:, :dims]
  return x / np.linalg.norm(x, axis=1, keepdims=True)


rng = np.random.default_rng(0)
requetes = rng.choice(len(vecteurs), 200, replace=False)


def voisins(x: np.ndarray) -> np.ndarray:
  scores = x[requetes] @ x.T
  scores[
    np.arange(len(requetes)), requetes
  ] = -np.inf  # un passage n'est pas son propre voisin
  return np.argsort(-scores, axis=1)[:, :10]


def compare(nom: str, x: np.ndarray, octets: float) -> None:
  trouves = voisins(x)
  part = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(reference, trouves)])
  print(
    f"{nom:18} : {part:4.0%} des voisins retrouvés, {octets:4.0f} octets par vecteur,"
    f" {octets * 1e6 / 1e9:.1f} Go par million de passages"
  )


reference = voisins(vecteurs)
compare("1024 dimensions", vecteurs, 1024 * 4)
for dims in (512, 256, 128, 64):
  compare(f"{dims} dimensions", tronque(vecteurs, dims), dims * 4)
compare("1024 bits (signes)", np.where(vecteurs > 0, 1.0, -1.0), 1024 / 8)

# Le recouvrement mesure la stabilité, pas la qualité : deux voisins différents peuvent
# être aussi bons. Seul un jeu de requêtes jugées la mesure (TP suivant).
# Les signes des 1024 dimensions retrouvent presque autant de voisins (64 %) que 256
# dimensions en float32 (67 %), pour 128 octets par vecteur au lieu de 1 024 : 8 fois moins,
# et 32 fois moins que le vecteur entier (4 096 octets). D'où l'usage : chercher avec les
# vecteurs binaires, puis reclasser les meilleurs avec les vrais vecteurs.